# End-to-End Credit Risk Analysis
## Notebook 2 — ML Model Training + Credit Score Generation (Google Colab Version)
**Input:** cleaned_data.csv (from Google Drive)  
**Output:** scored_applicants.csv (saved to Google Drive)

---
## Step 0 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

project_path = '/content/drive/MyDrive/credit_risk_project'
data_path = f'{project_path}/data'

os.makedirs(data_path, exist_ok=True)
print('Project folder ready at:', project_path)

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

---
## Step 2 — Load Cleaned Data from Drive

In [ ]:
cleaned_file = f'{data_path}/cleaned_data.csv'

if not os.path.exists(cleaned_file):
    # Fallback: allow manual upload if Drive copy is missing
    from google.colab import files
    print('cleaned_data.csv not found in Drive. Please upload it now...')
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, cleaned_file)

df = pd.read_csv(cleaned_file)

print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

---
## Step 3 — Define Features & Target

In [ ]:
target = 'loan_status'

X = df.drop(columns=[target])
y = df[target]

print('Features shape:', X.shape)
print('Target distribution:')
print(y.value_counts())
print('\nDefault Rate:', round(y.mean() * 100, 2), '%')

---
## Step 4 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

---
## Step 5 — Model 1: Logistic Regression (Baseline)

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

lr_accuracy = accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_proba)

print('=== Logistic Regression ===')
print(f'Accuracy : {lr_accuracy * 100:.2f}%')
print(f'AUC Score: {lr_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred))

In [ ]:
plt.figure(figsize=(6, 4))
cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix — Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## Step 6 — Model 2: Random Forest (Improved)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

print('=== Random Forest ===')
print(f'Accuracy : {rf_accuracy * 100:.2f}%')
print(f'AUC Score: {rf_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred))

In [ ]:
plt.figure(figsize=(6, 4))
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix — Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## Step 7 — Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy (%)': [round(lr_accuracy * 100, 2), round(rf_accuracy * 100, 2)],
    'AUC Score': [round(lr_auc, 4), round(rf_auc, 4)]
})
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Logistic Regression', 'Random Forest'],
            [lr_accuracy * 100, rf_accuracy * 100],
            color=['steelblue', 'seagreen'])
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 100)

axes[1].bar(['Logistic Regression', 'Random Forest'],
            [lr_auc, rf_auc],
            color=['steelblue', 'seagreen'])
axes[1].set_title('AUC Score Comparison')
axes[1].set_ylabel('AUC Score')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)

plt.figure(figsize=(8, 5))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={lr_auc:.3f})', color='steelblue')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})', color='seagreen')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — Feature Importance (Random Forest)

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(10), palette='viridis')
plt.title('Top 10 Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
print(feature_importance.head())

---
## Step 9 — Credit Score Generation (300–900 Scale)

In [ ]:
full_proba = rf_model.predict_proba(X)[:, 0]  # probability of NOT defaulting

credit_scores = (300 + (full_proba * 600)).astype(int)

print('Credit Score Stats:')
print(f'  Min  : {credit_scores.min()}')
print(f'  Max  : {credit_scores.max()}')
print(f'  Mean : {credit_scores.mean():.0f}')
print(f'  Median: {np.median(credit_scores):.0f}')

In [ ]:
def assign_risk_category(score):
    if score >= 800:
        return 'Very Safe'
    elif score >= 700:
        return 'Safe'
    elif score >= 600:
        return 'Moderate Risk'
    else:
        return 'High Risk'

def assign_decision(score):
    if score > 750:
        return 'Approved'
    elif score >= 600:
        return 'Review'
    else:
        return 'Rejected'

df_scored = df.copy()
df_scored['credit_score'] = credit_scores
df_scored['risk_category'] = df_scored['credit_score'].apply(assign_risk_category)
df_scored['decision'] = df_scored['credit_score'].apply(assign_decision)

print('Sample scored applicants:')
df_scored[['credit_score', 'risk_category', 'decision']].head(10)

---
## Step 10 — Score Distribution & Decision Analysis

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df_scored['credit_score'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(750, color='green', linestyle='--', label='Approve threshold (750)')
plt.axvline(600, color='red', linestyle='--', label='Reject threshold (600)')
plt.title('Credit Score Distribution')
plt.xlabel('Credit Score')
plt.ylabel('Number of Applicants')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
decision_counts = df_scored['decision'].value_counts()
decision_pct = (decision_counts / len(df_scored) * 100).round(2)

print('=== Decision Summary ===')
for decision, count in decision_counts.items():
    print(f'{decision:10}: {count:6} applicants ({decision_pct[decision]}%)')

plt.figure(figsize=(6, 6))
colors = {'Approved': 'seagreen', 'Review': 'goldenrod', 'Rejected': 'tomato'}
plt.pie(
    decision_counts,
    labels=decision_counts.index,
    autopct='%1.1f%%',
    colors=[colors[d] for d in decision_counts.index],
    startangle=90
)
plt.title('Approval Decision Distribution')
plt.tight_layout()
plt.show()

In [ ]:
risk_counts = df_scored['risk_category'].value_counts()

plt.figure(figsize=(8, 4))
risk_counts.plot(kind='bar',
                 color=['seagreen', 'steelblue', 'goldenrod', 'tomato'])
plt.title('Risk Category Distribution')
plt.xlabel('Risk Category')
plt.ylabel('Number of Applicants')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## Step 11 — Save Final Scored Dataset to Google Drive

In [ ]:
output_file = f'{data_path}/scored_applicants.csv'
df_scored.to_csv(output_file, index=False)

if os.path.exists(output_file):
    print('scored_applicants.csv saved successfully to Google Drive!')
    print('Location:', output_file)
else:
    print('Save failed — check Drive permissions.')

print('\n=== Final Summary ===')
print(f'Total Applicants  : {len(df_scored)}')
print(f'Logistic Reg Acc  : {lr_accuracy * 100:.2f}%')
print(f'Random Forest Acc : {rf_accuracy * 100:.2f}%')
print(f'Random Forest AUC : {rf_auc:.4f}')
print(f'Approval Rate     : {decision_pct.get("Approved", 0)}%')
print(f'Review Rate       : {decision_pct.get("Review", 0)}%')
print(f'Rejection Rate    : {decision_pct.get("Rejected", 0)}%')
print('\nDay 2 Complete! Ready for SQL + Excel + Power BI.')

---
### Optional — Also download a local copy to your computer

In [ ]:
from google.colab import files
files.download(output_file)